# STIR-Net V1 — 23 notebook-only spatial-supervision patch experiment

This notebook tests the two spatial additions supported by Notebooks 20–22 **without modifying repository source files**.

The current repository already gives us the exact hooks we need:

```text
SpatialEncoder
    ↓
E3
    ↓
SpatialDecoder.decode_to_e2(...)
    ↓
E2
    ↓
SpatialDecoder.decode_from_e2(...)
    ↓
D1 → D0
    ↓
DenseAuxiliaryHeads
```

The current dense target/loss path supervises:

```text
foreground
center heatmap
general boundary
```

but does not separately supervise internal cell-cell boundaries or cell identity.

Notebook 23 patches those two missing training signals locally.

---

## Patch A — internal cell-cell boundary supervision

Keep the current general boundary loss unchanged and add a second independently reduced loss whose positive class is only:

```text
GT cell A ↔ GT cell B
```

within foreground.

This is the mechanism already validated twice in the micro experiments.

---

## Patch B — instance-discriminative spatial supervision

Add notebook-local sampled `1×1×1`-equivalent projections at **E2 and D1**.

For each imperfect current connected component that contains ≥2 GT cells:

- voxels from one GT cell are pulled toward that cell's embedding centroid;
- centroids of different GT cells inside the **same current merged component** are pushed apart.

This does **not** force the positive feature centroid to become a mask vector. It shapes the spatial representation itself.

Only sampled voxels are projected, so we do not allocate dense E2/D1 embedding volumes.

---

## Four independent arms

All arms start from the exact same Notebook-12 step-30 spatial checkpoint.

1. `baseline`
2. `internal_boundary`
3. `instance_embedding`
4. `combined`

Each arm runs only **5 spatial-only steps**.

No temporal branch, co-reasoning, query decoder, Hungarian matching, or native rendering is executed.

The actual encoder, `stage_e2`, `stage_e1`, `stage_e0`, and dense heads are trainable, matching the spatial-dense curriculum intent.

---

## Decisive acceptance metrics

The fix is useful only if the **raw spatial representation** improves, not merely the new loss scalar.

We measure:

- source-9 internal-boundary AUC;
- source-9 internal-boundary recall;
- free learned linear mask-vector held-out AUC at E2, D1, D0;
- worst-cell held-out AUC;
- foreground hard Dice;
- original dense objective;
- embedding intra/inter-centroid geometry;
- gradient paths into encoder / E2 decoder / D1 decoder.

### Desired result

```text
internal_boundary:
    boundary AUC ↑

instance_embedding:
    E2/D1 free-vector AUC ↑
    worst-cell AUC ↑

combined:
    both improvements together
    foreground remains stable
```

If the combined arm passes those gates, we have enough evidence to implement the same additions in the model source.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
import copy
import gc
import json
import math
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from scipy import ndimage as ndi
from scipy.stats import rankdata

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.coordinates import resize_label_map_nearest
from learned.stirnet.training.checkpoint import load_checkpoint

# -------------------------------------------------------------------------
# Experiment configuration
# -------------------------------------------------------------------------

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

TRAIN_STEPS = 5
EVAL_STEPS = {0, 1, 3, 5}
LINEAR_PROBE_STEPS = {0, 3, 5}

MICRO_LR = 2e-4

# Notebook-only patch weights.
LAMBDA_INTERNAL_BOUNDARY = 1.0
LAMBDA_INSTANCE_E2 = 0.75
LAMBDA_INSTANCE_D1 = 0.75

# Small sampled embedding head.
INSTANCE_EMBED_DIM = 8
INSTANCE_SAMPLES_PER_CELL = 96

# Normalized-embedding discriminative margins.
INSTANCE_VARIANCE_MARGIN = 0.20
INSTANCE_SEPARATION_MARGIN = 0.90

# Internal-boundary balanced sample.
MAX_INTERNAL_POS = 8192

# Free linear probe.
FREE_VECTOR_STEPS = 80
FREE_VECTOR_LR = 5e-2
FREE_VECTOR_WEIGHT_DECAY = 1e-4

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

STEP30_CHECKPOINT = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "first_overfit"
    / "12_staged_same_sample"
    / "checkpoint_spatial_dense.pt"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "targeted"
    / "23_spatial_supervision_patch"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook 23 requires CUDA.")

if not STEP30_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Exact step-30 checkpoint not found:\n"
        f"{STEP30_CHECKPOINT}"
    )

device = torch.device("cuda")
cfg = _reduced_config()

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Checkpoint :", STEP30_CHECKPOINT)
print("Run dir    :", RUN_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("Spatial channels:", cfg.spatial.channels)
print("Embedding dim   :", INSTANCE_EMBED_DIM)
print("Train steps     :", TRAIN_STEPS)
print("LR              :", MICRO_LR)

# 1. Load the same full all-cell scene

There is no isolated-cell patch training here. The complete prepared scene is retained.

In [ ]:
batch, sample = build_real_batch(DATA_DIR)

target = batch["targets"][0]
targets = batch["targets"]

spatial_inputs = batch["spatial_inputs"].to(
    device=device,
    dtype=AMP_DTYPE,
    non_blocking=True,
)

spacing_um = batch["spacing_um"].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)

dref_tensor = batch["dref_um"].to(
    device=device,
    dtype=torch.float32,
    non_blocking=True,
)

spatial_padding_mask = batch.get("spatial_padding_mask")
if spatial_padding_mask is not None:
    spatial_padding_mask = spatial_padding_mask.to(
        device=device,
        non_blocking=True,
    )

current_labels_native = (
    batch["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

gt_labels_native = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)

spacing_native = (
    batch["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)

dref_um = float(batch["dref_um"][0])

source9_gt_ids = np.unique(
    gt_labels_native[
        current_labels_native == SOURCE_ID
    ]
)
source9_gt_ids = source9_gt_ids[
    source9_gt_ids > 0
].astype(int)

print(json.dumps(sample, indent=2, default=float))
print("Source-9 GT IDs:", source9_gt_ids.tolist())
print("Source-9 GT count:", len(source9_gt_ids))
print("dref_um:", dref_um)

# 2. Notebook-local internal cell-cell boundary target

The repository's general boundary target is left untouched.

This additional target marks only interfaces where both neighboring voxels are foreground and their GT instance IDs differ.

In [ ]:
def internal_instance_boundary(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    boundary = np.zeros_like(labels, dtype=bool)

    for axis in range(3):
        left = [slice(None)] * 3
        right = [slice(None)] * 3
        left[axis] = slice(0, -1)
        right[axis] = slice(1, None)

        a = labels[tuple(left)]
        b = labels[tuple(right)]

        diff = (
            (a > 0)
            & (b > 0)
            & (a != b)
        )

        boundary[tuple(left)] |= diff
        boundary[tuple(right)] |= diff

    return boundary


def physical_dilate(
    mask: np.ndarray,
    spacing,
    width_um: float = 1.0,
) -> np.ndarray:
    spacing = np.asarray(spacing, dtype=np.float64)
    radius = np.ceil(
        float(width_um) / spacing
    ).astype(int)

    zz, yy, xx = np.ogrid[
        -radius[0]:radius[0] + 1,
        -radius[1]:radius[1] + 1,
        -radius[2]:radius[2] + 1,
    ]

    structure = (
        (zz * spacing[0]) ** 2
        + (yy * spacing[1]) ** 2
        + (xx * spacing[2]) ** 2
        <= float(width_um) ** 2
    )

    return ndi.binary_dilation(
        mask,
        structure=structure,
    )


INTERNAL_RAW_NATIVE = internal_instance_boundary(
    gt_labels_native
)

INTERNAL_TARGET_NATIVE = physical_dilate(
    INTERNAL_RAW_NATIVE,
    spacing_native,
    width_um=1.0,
)

GENERAL_BOUNDARY_NATIVE = (
    torch.as_tensor(target["boundary"])
    .cpu()
    .numpy()
    > 0.5
)

print(
    "General boundary positives:",
    int(GENERAL_BOUNDARY_NATIVE.sum()),
)
print(
    "Internal cell-cell positives:",
    int(
        (
            INTERNAL_TARGET_NATIVE
            & GENERAL_BOUNDARY_NATIVE
        ).sum()
    ),
)
print(
    "Internal fraction of general positive target:",
    float(
        (
            INTERNAL_TARGET_NATIVE
            & GENERAL_BOUNDARY_NATIVE
        ).sum()
        / max(
            GENERAL_BOUNDARY_NATIVE.sum(),
            1,
        )
    ),
)

# 3. Discover E2 / D1 / D0 shapes from the exact step-30 model

This also confirms the notebook patch is operating on the current repository modules, not a reimplemented CNN.

In [ ]:
shape_model = StirNet(cfg).to(device)

load_info = load_checkpoint(
    STEP30_CHECKPOINT,
    shape_model,
    optimizer=None,
    scheduler=None,
    scaler=None,
    map_location="cpu",
    strict=True,
    migrate_history=True,
)

if int(load_info.get("step", -1)) != 30:
    raise RuntimeError(
        "Expected checkpoint payload step 30."
    )

shape_model.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=AMP_DTYPE,
):
    acq = shape_model.acquisition(
        spacing_um,
        dref_tensor,
    )

    pyramid = shape_model.encoder(
        spatial_inputs,
        spacing_um,
        acq,
        spatial_padding_mask,
    )

    e3 = pyramid.features[3]

    e2 = shape_model.decoder.decode_to_e2(
        e3,
        pyramid,
        acq,
    )

    d1, d0, _ = (
        shape_model.decoder.decode_from_e2(
            e2,
            pyramid,
            acq,
        )
    )

E2_SHAPE = tuple(int(v) for v in e2.shape[-3:])
D1_SHAPE = tuple(int(v) for v in d1.shape[-3:])
D0_SHAPE = tuple(int(v) for v in d0.shape[-3:])

E2_SPACING = (
    pyramid.spacings_um[2][0]
    .detach()
    .float()
    .cpu()
    .numpy()
)

D1_SPACING = (
    pyramid.spacings_um[1][0]
    .detach()
    .float()
    .cpu()
    .numpy()
)

D0_SPACING = (
    pyramid.spacings_um[0][0]
    .detach()
    .float()
    .cpu()
    .numpy()
)

print("E2:", E2_SHAPE, "spacing:", E2_SPACING)
print("D1:", D1_SHAPE, "spacing:", D1_SPACING)
print("D0:", D0_SHAPE, "spacing:", D0_SPACING)

del (
    shape_model,
    acq,
    pyramid,
    e3,
    e2,
    d1,
    d0,
)
gc.collect()
torch.cuda.empty_cache()

# 4. Resize integer labels to each feature level

In [ ]:
def labels_at_shape(
    labels_native: np.ndarray,
    shape,
) -> np.ndarray:
    return (
        resize_label_map_nearest(
            torch.from_numpy(
                labels_native.astype(
                    np.int32,
                    copy=False,
                )
            ),
            tuple(int(v) for v in shape),
        )
        .cpu()
        .numpy()
        .astype(np.int32, copy=False)
    )


GT_E2 = labels_at_shape(
    gt_labels_native,
    E2_SHAPE,
)
CURRENT_E2 = labels_at_shape(
    current_labels_native,
    E2_SHAPE,
)

GT_D1 = labels_at_shape(
    gt_labels_native,
    D1_SHAPE,
)
CURRENT_D1 = labels_at_shape(
    current_labels_native,
    D1_SHAPE,
)

GT_D0 = labels_at_shape(
    gt_labels_native,
    D0_SHAPE,
)
CURRENT_D0 = labels_at_shape(
    current_labels_native,
    D0_SHAPE,
)

INTERNAL_RAW_D0 = internal_instance_boundary(
    GT_D0
)

INTERNAL_TARGET_D0 = physical_dilate(
    INTERNAL_RAW_D0,
    D0_SPACING,
    width_um=1.0,
)

FOREGROUND_D0 = GT_D0 > 0

print(
    "Source-9 GT count E2/D1/D0:",
    len(
        np.unique(
            GT_E2[
                CURRENT_E2 == SOURCE_ID
            ]
        )[
            np.unique(
                GT_E2[
                    CURRENT_E2 == SOURCE_ID
                ]
            ) > 0
        ]
    ),
    len(
        np.unique(
            GT_D1[
                CURRENT_D1 == SOURCE_ID
            ]
        )[
            np.unique(
                GT_D1[
                    CURRENT_D1 == SOURCE_ID
                ]
            ) > 0
        ]
    ),
    len(
        np.unique(
            GT_D0[
                CURRENT_D0 == SOURCE_ID
            ]
        )[
            np.unique(
                GT_D0[
                    CURRENT_D0 == SOURCE_ID
                ]
            ) > 0
        ]
    ),
)

# 5. Fixed balanced internal-boundary sample

The extra boundary objective is independently normalized over a balanced subset of internal positives and foreground non-boundary negatives.

This prevents the internal signal from being diluted by the much larger outer-boundary population.

In [ ]:
def deterministic_choice(
    indices,
    count,
    seed,
):
    indices = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(indices) <= int(count):
        return indices

    rng = np.random.default_rng(int(seed))

    selected = rng.choice(
        indices,
        size=int(count),
        replace=False,
    )

    return np.sort(selected)


@dataclass
class BinarySample:
    indices: torch.Tensor
    targets: torch.Tensor


internal_positive = np.flatnonzero(
    INTERNAL_TARGET_D0.reshape(-1)
)

internal_negative = np.flatnonzero(
    (
        FOREGROUND_D0
        & ~INTERNAL_TARGET_D0
    ).reshape(-1)
)

internal_positive = deterministic_choice(
    internal_positive,
    min(
        len(internal_positive),
        MAX_INTERNAL_POS,
    ),
    SEED + 101,
)

internal_negative = deterministic_choice(
    internal_negative,
    len(internal_positive),
    SEED + 102,
)

internal_indices = np.concatenate(
    [
        internal_positive,
        internal_negative,
    ]
)

internal_targets = np.concatenate(
    [
        np.ones(
            len(internal_positive),
            dtype=np.float32,
        ),
        np.zeros(
            len(internal_negative),
            dtype=np.float32,
        ),
    ]
)

rng = np.random.default_rng(SEED + 103)
order = rng.permutation(len(internal_indices))

INTERNAL_SAMPLE = BinarySample(
    indices=torch.from_numpy(
        internal_indices[order]
    ).long(),
    targets=torch.from_numpy(
        internal_targets[order]
    ).float(),
)

print(
    "Internal-boundary training sample:",
    int(
        (
            INTERNAL_SAMPLE.targets == 1
        ).sum()
    ),
    "positive +",
    int(
        (
            INTERNAL_SAMPLE.targets == 0
        ).sum()
    ),
    "negative",
)

# 6. Fixed merged-component embedding groups

Only current connected components that overlap at least two GT cells generate the new identity loss.

This focuses supervision on exactly the failure mode STIR-Net is intended to correct.

In [ ]:
@dataclass
class EmbeddingCell:
    gt_id: int
    indices: torch.Tensor


@dataclass
class EmbeddingGroup:
    source_id: int
    cells: tuple


def build_embedding_groups(
    current_labels,
    gt_labels,
    *,
    samples_per_cell,
    seed,
):
    current_flat = np.asarray(
        current_labels
    ).reshape(-1)

    gt_flat = np.asarray(
        gt_labels
    ).reshape(-1)

    groups = []

    for source_id in np.unique(current_flat):
        source_id = int(source_id)

        if source_id <= 0:
            continue

        source_mask = (
            current_flat == source_id
        )

        gt_ids = np.unique(
            gt_flat[source_mask]
        )

        gt_ids = gt_ids[
            gt_ids > 0
        ]

        if len(gt_ids) < 2:
            continue

        cells = []

        for gt_id in gt_ids:
            gt_id = int(gt_id)

            positions = np.flatnonzero(
                source_mask
                & (
                    gt_flat == gt_id
                )
            )

            if len(positions) < 4:
                continue

            chosen = deterministic_choice(
                positions,
                min(
                    len(positions),
                    int(samples_per_cell),
                ),
                seed
                + 1009 * source_id
                + 37 * gt_id,
            )

            if len(chosen) < 4:
                continue

            cells.append(
                EmbeddingCell(
                    gt_id=gt_id,
                    indices=torch.from_numpy(
                        chosen
                    ).long(),
                )
            )

        if len(cells) >= 2:
            groups.append(
                EmbeddingGroup(
                    source_id=source_id,
                    cells=tuple(cells),
                )
            )

    return groups


EMBED_GROUPS_E2 = build_embedding_groups(
    CURRENT_E2,
    GT_E2,
    samples_per_cell=INSTANCE_SAMPLES_PER_CELL,
    seed=SEED + 1000,
)

EMBED_GROUPS_D1 = build_embedding_groups(
    CURRENT_D1,
    GT_D1,
    samples_per_cell=INSTANCE_SAMPLES_PER_CELL,
    seed=SEED + 2000,
)

print(
    "Merged groups E2:",
    [
        (
            group.source_id,
            [cell.gt_id for cell in group.cells],
        )
        for group in EMBED_GROUPS_E2
    ],
)

print(
    "Merged groups D1:",
    [
        (
            group.source_id,
            [cell.gt_id for cell in group.cells],
        )
        for group in EMBED_GROUPS_D1
    ],
)

print(
    "Source-9 E2 cells:",
    next(
        (
            [cell.gt_id for cell in group.cells]
            for group in EMBED_GROUPS_E2
            if group.source_id == SOURCE_ID
        ),
        [],
    ),
)

# 7. Source-9 linear-probe tasks

These are separate from the discriminative loss implementation.

For each source-9 GT cell:

```text
train half:
    learn a completely free affine mask vector

held-out half:
    measure AUC against sibling-cell voxels
```

The raw E2/D1/D0 features are probed. The auxiliary embedding head is **not** used for this metric.

That makes this the strongest acceptance signal: the actual spatial representation must become more linearly cell-separable.

In [ ]:
@dataclass
class LinearProbeTask:
    gt_id: int
    train_pos: torch.Tensor
    train_neg: torch.Tensor
    eval_pos: torch.Tensor
    eval_neg: torch.Tensor


def build_linear_probe_tasks(
    current_labels,
    gt_labels,
    *,
    source_id,
    seed,
    max_per_split=128,
):
    current_flat = np.asarray(
        current_labels
    ).reshape(-1)

    gt_flat = np.asarray(
        gt_labels
    ).reshape(-1)

    source_mask = (
        current_flat == int(source_id)
    )

    gt_ids = np.unique(
        gt_flat[source_mask]
    )

    gt_ids = gt_ids[
        gt_ids > 0
    ]

    tasks = []

    for gt_id in gt_ids:
        gt_id = int(gt_id)

        positive = np.flatnonzero(
            source_mask
            & (
                gt_flat == gt_id
            )
        )

        negative = np.flatnonzero(
            source_mask
            & (
                gt_flat > 0
            )
            & (
                gt_flat != gt_id
            )
        )

        if (
            len(positive) < 8
            or len(negative) < 8
        ):
            continue

        rng_pos = np.random.default_rng(
            seed + 101 * gt_id
        )
        rng_neg = np.random.default_rng(
            seed + 103 * gt_id
        )

        positive = positive[
            rng_pos.permutation(
                len(positive)
            )
        ]

        negative = negative[
            rng_neg.permutation(
                len(negative)
            )
        ]

        pos_count = min(
            int(max_per_split),
            len(positive) // 2,
        )

        neg_count = min(
            int(max_per_split),
            len(negative) // 2,
        )

        if (
            pos_count < 4
            or neg_count < 4
        ):
            continue

        tasks.append(
            LinearProbeTask(
                gt_id=gt_id,
                train_pos=torch.from_numpy(
                    positive[:pos_count]
                ).long(),
                train_neg=torch.from_numpy(
                    negative[:neg_count]
                ).long(),
                eval_pos=torch.from_numpy(
                    positive[
                        pos_count:
                        2 * pos_count
                    ]
                ).long(),
                eval_neg=torch.from_numpy(
                    negative[
                        neg_count:
                        2 * neg_count
                    ]
                ).long(),
            )
        )

    return tasks


PROBE_TASKS_E2 = build_linear_probe_tasks(
    CURRENT_E2,
    GT_E2,
    source_id=SOURCE_ID,
    seed=SEED + 3000,
)

PROBE_TASKS_D1 = build_linear_probe_tasks(
    CURRENT_D1,
    GT_D1,
    source_id=SOURCE_ID,
    seed=SEED + 4000,
)

PROBE_TASKS_D0 = build_linear_probe_tasks(
    CURRENT_D0,
    GT_D0,
    source_id=SOURCE_ID,
    seed=SEED + 5000,
)

print(
    "Probe task counts E2/D1/D0:",
    len(PROBE_TASKS_E2),
    len(PROBE_TASKS_D1),
    len(PROBE_TASKS_D0),
)

# 8. Notebook-only patched model

This class uses the repository's real spatial modules and adds only two sampled linear projections.

It does not monkey-patch files on disk.

In [ ]:
class SpatialSupervisionPatch(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        self.base = StirNet(cfg)

        c0, c1, c2, _ = cfg.spatial.channels

        # Sampled nn.Linear is equivalent to a 1x1x1 Conv3d at selected voxels.
        self.instance_proj_e2 = nn.Linear(
            c2,
            INSTANCE_EMBED_DIM,
            bias=False,
        )

        self.instance_proj_d1 = nn.Linear(
            c1,
            INSTANCE_EMBED_DIM,
            bias=False,
        )

    def load_step30(self):
        return load_checkpoint(
            STEP30_CHECKPOINT,
            self.base,
            optimizer=None,
            scheduler=None,
            scaler=None,
            map_location="cpu",
            strict=True,
            migrate_history=True,
        )

    def enable_spatial_training_only(self):
        # Freeze everything first.
        for parameter in self.base.parameters():
            parameter.requires_grad_(False)

        # Match the current spatial_dense trainable groups.
        for module in (
            self.base.acquisition,
            self.base.encoder,
            self.base.decoder,
            self.base.dense_heads,
        ):
            for parameter in module.parameters():
                parameter.requires_grad_(True)

        for parameter in self.instance_proj_e2.parameters():
            parameter.requires_grad_(True)

        for parameter in self.instance_proj_d1.parameters():
            parameter.requires_grad_(True)

    def forward_spatial(self):
        acq = self.base.acquisition(
            spacing_um,
            dref_tensor,
        )

        pyramid = self.base.encoder(
            spatial_inputs,
            spacing_um,
            acq,
            spatial_padding_mask,
        )

        # spatial_dense bypasses co-reasoning:
        e3 = pyramid.features[3]

        e2 = self.base.decoder.decode_to_e2(
            e3,
            pyramid,
            acq,
        )

        d1, d0, mask_features = (
            self.base.decoder.decode_from_e2(
                e2,
                pyramid,
                acq,
            )
        )

        dense = self.base.dense_heads(
            d0
        )

        return {
            "e2": e2,
            "d1": d1,
            "d0": d0,
            "mask_features": mask_features,
            "dense": dense,
        }


criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
    cfg.training,
).to(device)

# 9. Patch-loss implementations

In [ ]:
def gather_feature_rows(
    feature,
    indices_cpu,
):
    # feature [1,C,Z,Y,X] -> selected [N,C]
    flat = (
        feature[0]
        .flatten(1)
        .transpose(0, 1)
    )

    indices = indices_cpu.to(
        device=feature.device,
        non_blocking=True,
    )

    return flat[indices]


def balanced_binary_loss(
    logits,
    targets,
):
    logits = logits.float()

    targets = targets.to(
        device=logits.device,
        dtype=torch.float32,
        non_blocking=True,
    )

    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
    )

    probability = logits.sigmoid()

    intersection = (
        probability * targets
    ).sum()

    dice_loss = (
        1.0
        - (
            2.0 * intersection
            + 1e-6
        )
        / (
            probability.sum()
            + targets.sum()
            + 1e-6
        )
    )

    return bce + dice_loss


def internal_boundary_patch_loss(
    dense,
):
    logits_flat = (
        dense["boundary_logits"][0, 0]
        .reshape(-1)
    )

    indices = INTERNAL_SAMPLE.indices.to(
        device=logits_flat.device,
        non_blocking=True,
    )

    logits = logits_flat[indices]

    return balanced_binary_loss(
        logits,
        INTERNAL_SAMPLE.targets,
    )


def discriminative_instance_loss(
    feature,
    groups,
    projector,
):
    variance_terms = []
    separation_terms = []

    for group in groups:
        cell_embeddings = []
        centroids = []

        for cell in group.cells:
            rows = gather_feature_rows(
                feature,
                cell.indices,
            ).float()

            embedding = projector(
                rows
            )

            # Keep geometry bounded and make margins interpretable.
            embedding = F.normalize(
                embedding,
                dim=-1,
            )

            centroid = embedding.mean(
                dim=0
            )

            centroid = F.normalize(
                centroid,
                dim=-1,
            )

            distance_to_centroid = (
                torch.linalg.vector_norm(
                    embedding
                    - centroid[None, :],
                    dim=-1,
                )
            )

            variance_terms.append(
                F.relu(
                    distance_to_centroid
                    - float(
                        INSTANCE_VARIANCE_MARGIN
                    )
                )
                .square()
                .mean()
            )

            cell_embeddings.append(
                embedding
            )

            centroids.append(
                centroid
            )

        if len(centroids) >= 2:
            centroids = torch.stack(
                centroids,
                dim=0,
            )

            pairwise = torch.cdist(
                centroids,
                centroids,
                p=2,
            )

            upper = torch.triu(
                torch.ones_like(
                    pairwise,
                    dtype=torch.bool,
                ),
                diagonal=1,
            )

            pair_distance = pairwise[
                upper
            ]

            separation_terms.append(
                F.relu(
                    float(
                        INSTANCE_SEPARATION_MARGIN
                    )
                    - pair_distance
                )
                .square()
                .mean()
            )

    zero = feature.sum() * 0.0

    variance = (
        torch.stack(
            variance_terms
        ).mean()
        if variance_terms
        else zero
    )

    separation = (
        torch.stack(
            separation_terms
        ).mean()
        if separation_terms
        else zero
    )

    return {
        "loss": variance + separation,
        "variance": variance,
        "separation": separation,
    }


def existing_dense_objective(
    dense,
):
    foreground, center, boundary = (
        criterion._dense_losses(
            dense,
            targets,
        )
    )

    total = (
        float(cfg.losses.foreground)
        * foreground
        + float(cfg.losses.center_heatmap)
        * center
        + float(cfg.losses.boundary)
        * boundary
    )

    return {
        "total": total,
        "foreground": foreground,
        "center": center,
        "boundary": boundary,
    }

# 10. Diagnostic metrics

In [ ]:
def auc_from_scores(
    scores,
    labels,
):
    scores = np.asarray(
        scores,
        dtype=np.float64,
    )

    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    positive = labels == 1
    negative = labels == 0

    if (
        positive.sum() == 0
        or negative.sum() == 0
    ):
        return float("nan")

    ranks = rankdata(scores)

    n_positive = int(
        positive.sum()
    )

    n_negative = int(
        negative.sum()
    )

    return float(
        (
            ranks[positive].sum()
            - n_positive
            * (
                n_positive + 1
            )
            / 2
        )
        / (
            n_positive
            * n_negative
        )
    )


# Fixed source-9 D0 index sets.
S9_POS_D0 = np.flatnonzero(
    (
        (CURRENT_D0 == SOURCE_ID)
        & INTERNAL_RAW_D0
    ).reshape(-1)
)

S9_NEG_D0 = np.flatnonzero(
    (
        (CURRENT_D0 == SOURCE_ID)
        & ~INTERNAL_RAW_D0
    ).reshape(-1)
)

S9_POS_D0 = torch.from_numpy(
    S9_POS_D0
).long()

S9_NEG_D0 = torch.from_numpy(
    S9_NEG_D0
).long()


def source9_boundary_metrics(
    boundary_logits,
):
    flat = boundary_logits[
        0,
        0
    ].reshape(-1)

    pos = (
        flat[
            S9_POS_D0.to(
                flat.device
            )
        ]
        .detach()
        .float()
        .cpu()
        .sigmoid()
        .numpy()
    )

    neg = (
        flat[
            S9_NEG_D0.to(
                flat.device
            )
        ]
        .detach()
        .float()
        .cpu()
        .sigmoid()
        .numpy()
    )

    scores = np.concatenate(
        [
            pos,
            neg,
        ]
    )

    labels = np.concatenate(
        [
            np.ones(
                len(pos),
                dtype=np.int64,
            ),
            np.zeros(
                len(neg),
                dtype=np.int64,
            ),
        ]
    )

    return {
        "source9_internal_auc": (
            auc_from_scores(
                scores,
                labels,
            )
        ),
        "source9_internal_recall_0p5": float(
            (
                pos >= 0.5
            ).mean()
        ),
        "source9_internal_probability": float(
            pos.mean()
        ),
        "source9_nonboundary_probability": float(
            neg.mean()
        ),
    }


GT_FOREGROUND_TENSOR = (
    torch.from_numpy(
        (GT_D0 > 0)
        .astype(np.bool_)
    )
)


def foreground_hard_dice(
    foreground_logits,
):
    prediction = (
        foreground_logits[
            0,
            0
        ]
        .detach()
        .sigmoid()
        >= 0.5
    )

    target_fg = GT_FOREGROUND_TENSOR.to(
        device=prediction.device,
        non_blocking=True,
    )

    intersection = (
        prediction
        & target_fg
    ).sum().float()

    denominator = (
        prediction.sum().float()
        + target_fg.sum().float()
    )

    return float(
        (
            (
                2.0 * intersection
                + 1e-6
            )
            / (
                denominator
                + 1e-6
            )
        )
        .cpu()
    )


def embedding_geometry_metrics(
    feature,
    groups,
    projector,
    *,
    source_id=SOURCE_ID,
):
    selected = [
        group
        for group in groups
        if group.source_id
        == int(source_id)
    ]

    if not selected:
        return {
            "intra_distance": float("nan"),
            "inter_centroid_distance": float("nan"),
        }

    group = selected[0]

    intra = []
    centroids = []

    for cell in group.cells:
        rows = gather_feature_rows(
            feature,
            cell.indices,
        ).float()

        embedding = F.normalize(
            projector(rows),
            dim=-1,
        )

        centroid = F.normalize(
            embedding.mean(
                dim=0
            ),
            dim=-1,
        )

        intra.append(
            torch.linalg.vector_norm(
                embedding
                - centroid[None, :],
                dim=-1,
            ).mean()
        )

        centroids.append(
            centroid
        )

    centroids = torch.stack(
        centroids,
        dim=0,
    )

    pairwise = torch.cdist(
        centroids,
        centroids,
    )

    upper = torch.triu(
        torch.ones_like(
            pairwise,
            dtype=torch.bool,
        ),
        diagonal=1,
    )

    return {
        "intra_distance": float(
            torch.stack(
                intra
            ).mean()
            .detach()
            .cpu()
        ),
        "inter_centroid_distance": float(
            pairwise[
                upper
            ].mean()
            .detach()
            .cpu()
        ),
    }

# 11. Free learned mask-vector probe

This is the same causal idea as the final Notebook-22 probe:

```text
freeze current feature map
learn only one free affine mask vector per source-9 GT cell
evaluate on held-out sibling voxels
```

AUC is the primary metric.

In [ ]:
def compact_probe_dataset(
    feature,
    tasks,
):
    datasets = []

    flat = (
        feature[0]
        .flatten(1)
        .transpose(0, 1)
    )

    for task in tasks:
        def rows(indices):
            return (
                flat[
                    indices.to(
                        flat.device
                    )
                ]
                .detach()
                .float()
                .cpu()
            )

        train_pos = rows(
            task.train_pos
        )

        train_neg = rows(
            task.train_neg
        )

        eval_pos = rows(
            task.eval_pos
        )

        eval_neg = rows(
            task.eval_neg
        )

        train_x = torch.cat(
            [
                train_pos,
                train_neg,
            ],
            dim=0,
        )

        train_y = torch.cat(
            [
                torch.ones(
                    len(train_pos)
                ),
                torch.zeros(
                    len(train_neg)
                ),
            ]
        )

        eval_x = torch.cat(
            [
                eval_pos,
                eval_neg,
            ],
            dim=0,
        )

        eval_y = torch.cat(
            [
                torch.ones(
                    len(eval_pos)
                ),
                torch.zeros(
                    len(eval_neg)
                ),
            ]
        )

        mean = train_x.mean(
            dim=0,
            keepdim=True,
        )

        std = train_x.std(
            dim=0,
            keepdim=True,
        ).clamp_min(
            1e-4
        )

        datasets.append({
            "gt_id": task.gt_id,
            "train_x": (
                train_x - mean
            ) / std,
            "train_y": train_y,
            "eval_x": (
                eval_x - mean
            ) / std,
            "eval_y": eval_y,
        })

    return datasets


def free_vector_probe(
    feature,
    tasks,
):
    """
    Fit tiny free affine mask vectors on frozen feature rows.

    evaluate_snapshot() runs under @torch.no_grad(), so this helper must
    explicitly re-enable autograd for the temporary probe parameters only.
    The STIR-Net feature tensors remain detached/frozen.
    """
    datasets = compact_probe_dataset(
        feature,
        tasks,
    )

    if not datasets:
        return {
            "mean_auc": float("nan"),
            "median_auc": float("nan"),
            "min_auc": float("nan"),
            "per_cell": {},
        }

    channel_count = int(
        datasets[0]["train_x"].shape[1]
    )

    task_count = len(datasets)

    with torch.enable_grad():
        weights = nn.Parameter(
            torch.zeros(
                (
                    task_count,
                    channel_count,
                ),
                device=device,
                dtype=torch.float32,
            )
        )

        bias = nn.Parameter(
            torch.zeros(
                task_count,
                device=device,
                dtype=torch.float32,
            )
        )

        optimizer = torch.optim.AdamW(
            [
                weights,
                bias,
            ],
            lr=float(
                FREE_VECTOR_LR
            ),
            weight_decay=float(
                FREE_VECTOR_WEIGHT_DECAY
            ),
        )

        train_x = [
            item["train_x"].to(
                device=device,
                dtype=torch.float32,
            )
            for item in datasets
        ]

        train_y = [
            item["train_y"].to(
                device=device,
                dtype=torch.float32,
            )
            for item in datasets
        ]

        for _ in range(
            FREE_VECTOR_STEPS
        ):
            optimizer.zero_grad(
                set_to_none=True
            )

            losses = []

            for task_index in range(
                task_count
            ):
                logits = (
                    train_x[task_index]
                    @ weights[task_index]
                    + bias[task_index]
                )

                losses.append(
                    F.binary_cross_entropy_with_logits(
                        logits,
                        train_y[task_index],
                    )
                )

            loss = torch.stack(
                losses
            ).mean()

            if not loss.requires_grad:
                raise RuntimeError(
                    "Free-vector probe loss has no gradient graph. "
                    "The temporary probe must run inside torch.enable_grad()."
                )

            loss.backward()
            optimizer.step()

    aucs = []
    per_cell = {}

    with torch.no_grad():
        for task_index, item in enumerate(
            datasets
        ):
            eval_x = item[
                "eval_x"
            ].to(
                device=device,
                dtype=torch.float32,
            )

            eval_y = item[
                "eval_y"
            ].numpy().astype(
                np.int64
            )

            logits = (
                eval_x
                @ weights[task_index]
                + bias[task_index]
            )

            auc = auc_from_scores(
                logits.detach()
                .cpu()
                .numpy(),
                eval_y,
            )

            aucs.append(auc)

            per_cell[
                int(
                    item["gt_id"]
                )
            ] = float(auc)

    del (
        weights,
        bias,
        optimizer,
        train_x,
        train_y,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "mean_auc": float(
            np.mean(aucs)
        ),
        "median_auc": float(
            np.median(aucs)
        ),
        "min_auc": float(
            np.min(aucs)
        ),
        "per_cell": per_cell,
    }

# 12. Independent experiment arms

In [ ]:
ARM_SPECS = {
    "baseline": {
        "internal_boundary": False,
        "instance_embedding": False,
    },
    "internal_boundary": {
        "internal_boundary": True,
        "instance_embedding": False,
    },
    "instance_embedding": {
        "internal_boundary": False,
        "instance_embedding": True,
    },
    "combined": {
        "internal_boundary": True,
        "instance_embedding": True,
    },
}

display(
    pd.DataFrame(
        ARM_SPECS
    ).T
)

# 13. Objective and optimizer construction

In [ ]:
def patch_objective(
    model,
    outputs,
    spec,
):
    dense_loss = existing_dense_objective(
        outputs["dense"]
    )

    zero = (
        dense_loss["total"]
        * 0.0
    )

    internal = zero

    e2_identity = {
        "loss": zero,
        "variance": zero,
        "separation": zero,
    }

    d1_identity = {
        "loss": zero,
        "variance": zero,
        "separation": zero,
    }

    if spec[
        "internal_boundary"
    ]:
        internal = (
            internal_boundary_patch_loss(
                outputs["dense"]
            )
        )

    if spec[
        "instance_embedding"
    ]:
        e2_identity = (
            discriminative_instance_loss(
                outputs["e2"],
                EMBED_GROUPS_E2,
                model.instance_proj_e2,
            )
        )

        d1_identity = (
            discriminative_instance_loss(
                outputs["d1"],
                EMBED_GROUPS_D1,
                model.instance_proj_d1,
            )
        )

    total = (
        dense_loss["total"]
        + float(
            LAMBDA_INTERNAL_BOUNDARY
        )
        * internal
        + float(
            LAMBDA_INSTANCE_E2
        )
        * e2_identity["loss"]
        + float(
            LAMBDA_INSTANCE_D1
        )
        * d1_identity["loss"]
    )

    return {
        "loss": total,
        "dense_total": (
            dense_loss["total"]
        ),
        "foreground": (
            dense_loss["foreground"]
        ),
        "center": (
            dense_loss["center"]
        ),
        "boundary": (
            dense_loss["boundary"]
        ),
        "internal_boundary": (
            internal
        ),
        "e2_identity": (
            e2_identity["loss"]
        ),
        "e2_variance": (
            e2_identity["variance"]
        ),
        "e2_separation": (
            e2_identity["separation"]
        ),
        "d1_identity": (
            d1_identity["loss"]
        ),
        "d1_variance": (
            d1_identity["variance"]
        ),
        "d1_separation": (
            d1_identity["separation"]
        ),
    }


def trainable_parameters(
    model,
):
    parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    if not parameters:
        raise RuntimeError(
            "No trainable parameters."
        )

    return parameters


def subsystem_grad_norm(
    module,
):
    total = 0.0
    finite = True

    for parameter in module.parameters():
        if parameter.grad is None:
            continue

        grad = parameter.grad.detach()

        finite = (
            finite
            and bool(
                torch.isfinite(
                    grad
                ).all()
            )
        )

        total += float(
            grad.float()
            .square()
            .sum()
            .detach()
            .cpu()
        )

    return (
        math.sqrt(
            max(
                total,
                0.0,
            )
        ),
        finite,
    )

# 14. Evaluation

The free-vector probe is intentionally run only at steps 0, 3, and 5 because it is a nested diagnostic optimization.

In [ ]:
@torch.no_grad()
def evaluate_snapshot(
    model,
    *,
    arm,
    step,
    run_linear_probe,
):
    model.eval()

    with torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = (
            model.forward_spatial()
        )

        dense_eval = (
            existing_dense_objective(
                outputs["dense"]
            )
        )

    boundary_metrics = (
        source9_boundary_metrics(
            outputs["dense"][
                "boundary_logits"
            ]
        )
    )

    fg_dice = foreground_hard_dice(
        outputs["dense"][
            "foreground_logits"
        ]
    )

    e2_geometry = (
        embedding_geometry_metrics(
            outputs["e2"],
            EMBED_GROUPS_E2,
            model.instance_proj_e2,
        )
    )

    d1_geometry = (
        embedding_geometry_metrics(
            outputs["d1"],
            EMBED_GROUPS_D1,
            model.instance_proj_d1,
        )
    )

    result = {
        "arm": arm,
        "step": int(step),
        "dense_total": float(
            dense_eval["total"]
            .detach()
            .cpu()
        ),
        "foreground_loss": float(
            dense_eval["foreground"]
            .detach()
            .cpu()
        ),
        "center_loss": float(
            dense_eval["center"]
            .detach()
            .cpu()
        ),
        "boundary_loss": float(
            dense_eval["boundary"]
            .detach()
            .cpu()
        ),
        "foreground_hard_dice": (
            fg_dice
        ),
        **boundary_metrics,
        "e2_embedding_intra": (
            e2_geometry["intra_distance"]
        ),
        "e2_embedding_inter": (
            e2_geometry[
                "inter_centroid_distance"
            ]
        ),
        "d1_embedding_intra": (
            d1_geometry["intra_distance"]
        ),
        "d1_embedding_inter": (
            d1_geometry[
                "inter_centroid_distance"
            ]
        ),
    }

    if run_linear_probe:
        e2_probe = free_vector_probe(
            outputs["e2"],
            PROBE_TASKS_E2,
        )

        d1_probe = free_vector_probe(
            outputs["d1"],
            PROBE_TASKS_D1,
        )

        d0_probe = free_vector_probe(
            outputs["d0"],
            PROBE_TASKS_D0,
        )

        result.update({
            "e2_free_auc_mean": (
                e2_probe["mean_auc"]
            ),
            "e2_free_auc_median": (
                e2_probe["median_auc"]
            ),
            "e2_free_auc_min": (
                e2_probe["min_auc"]
            ),
            "d1_free_auc_mean": (
                d1_probe["mean_auc"]
            ),
            "d1_free_auc_median": (
                d1_probe["median_auc"]
            ),
            "d1_free_auc_min": (
                d1_probe["min_auc"]
            ),
            "d0_free_auc_mean": (
                d0_probe["mean_auc"]
            ),
            "d0_free_auc_median": (
                d0_probe["median_auc"]
            ),
            "d0_free_auc_min": (
                d0_probe["min_auc"]
            ),
            "e2_free_per_cell": json.dumps(
                e2_probe["per_cell"]
            ),
            "d1_free_per_cell": json.dumps(
                d1_probe["per_cell"]
            ),
            "d0_free_per_cell": json.dumps(
                d0_probe["per_cell"]
            ),
        })
    else:
        result.update({
            "e2_free_auc_mean": np.nan,
            "e2_free_auc_median": np.nan,
            "e2_free_auc_min": np.nan,
            "d1_free_auc_mean": np.nan,
            "d1_free_auc_median": np.nan,
            "d1_free_auc_min": np.nan,
            "d0_free_auc_mean": np.nan,
            "d0_free_auc_median": np.nan,
            "d0_free_auc_min": np.nan,
            "e2_free_per_cell": None,
            "d1_free_per_cell": None,
            "d0_free_per_cell": None,
        })

    del outputs

    gc.collect()
    torch.cuda.empty_cache()

    return result

# 15. Five-step independent micro-training runs

In [ ]:
def run_arm(
    arm,
    spec,
):
    torch.manual_seed(SEED + 9001)
    torch.cuda.manual_seed_all(SEED + 9001)

    model = SpatialSupervisionPatch(
        cfg
    ).to(device)

    load_info = model.load_step30()

    if int(
        load_info.get(
            "step",
            -1,
        )
    ) != 30:
        raise RuntimeError(
            "Arm did not load step 30."
        )

    model.enable_spatial_training_only()

    optimizer = torch.optim.AdamW(
        trainable_parameters(
            model
        ),
        lr=float(
            MICRO_LR
        ),
        weight_decay=float(
            cfg.training.weight_decay
        ),
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=True,
        init_scale=1024.0,
    )

    metric_rows = []
    train_rows = []

    arm_start = time.perf_counter()

    for step in range(
        TRAIN_STEPS + 1
    ):
        if step in EVAL_STEPS:
            print(
                f"  evaluating step {step} ..."
            )

            metric_rows.append(
                evaluate_snapshot(
                    model,
                    arm=arm,
                    step=step,
                    run_linear_probe=(
                        step
                        in LINEAR_PROBE_STEPS
                    ),
                )
            )

        if step == TRAIN_STEPS:
            break

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        torch.cuda.reset_peak_memory_stats()

        step_start = time.perf_counter()

        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        ):
            outputs = (
                model.forward_spatial()
            )

            objective = patch_objective(
                model,
                outputs,
                spec,
            )

        scaler.scale(
            objective["loss"]
        ).backward()

        scaler.unscale_(
            optimizer
        )

        encoder_grad, encoder_finite = (
            subsystem_grad_norm(
                model.base.encoder
            )
        )

        stage_e2_grad, stage_e2_finite = (
            subsystem_grad_norm(
                model.base.decoder.stage_e2
            )
        )

        stage_e1_grad, stage_e1_finite = (
            subsystem_grad_norm(
                model.base.decoder.stage_e1
            )
        )

        stage_e0_grad, stage_e0_finite = (
            subsystem_grad_norm(
                model.base.decoder.stage_e0
            )
        )

        dense_grad, dense_finite = (
            subsystem_grad_norm(
                model.base.dense_heads
            )
        )

        e2_proj_grad, e2_proj_finite = (
            subsystem_grad_norm(
                model.instance_proj_e2
            )
        )

        d1_proj_grad, d1_proj_finite = (
            subsystem_grad_norm(
                model.instance_proj_d1
            )
        )

        finite = all([
            encoder_finite,
            stage_e2_finite,
            stage_e1_finite,
            stage_e0_finite,
            dense_finite,
            e2_proj_finite,
            d1_proj_finite,
        ])

        total_grad_norm = float(
            torch.nn.utils.clip_grad_norm_(
                trainable_parameters(
                    model
                ),
                float(
                    cfg.training.max_grad_norm
                ),
            )
            .detach()
            .cpu()
        )

        scaler.step(optimizer)
        scaler.update()

        train_rows.append({
            "arm": arm,
            "from_step": int(step),
            "to_step": int(step + 1),
            "loss": float(
                objective["loss"]
                .detach()
                .cpu()
            ),
            "dense_total": float(
                objective["dense_total"]
                .detach()
                .cpu()
            ),
            "foreground": float(
                objective["foreground"]
                .detach()
                .cpu()
            ),
            "center": float(
                objective["center"]
                .detach()
                .cpu()
            ),
            "boundary": float(
                objective["boundary"]
                .detach()
                .cpu()
            ),
            "internal_boundary": float(
                objective[
                    "internal_boundary"
                ]
                .detach()
                .cpu()
            ),
            "e2_identity": float(
                objective["e2_identity"]
                .detach()
                .cpu()
            ),
            "e2_variance": float(
                objective["e2_variance"]
                .detach()
                .cpu()
            ),
            "e2_separation": float(
                objective["e2_separation"]
                .detach()
                .cpu()
            ),
            "d1_identity": float(
                objective["d1_identity"]
                .detach()
                .cpu()
            ),
            "d1_variance": float(
                objective["d1_variance"]
                .detach()
                .cpu()
            ),
            "d1_separation": float(
                objective["d1_separation"]
                .detach()
                .cpu()
            ),
            "encoder_grad_norm": float(
                encoder_grad
            ),
            "stage_e2_grad_norm": float(
                stage_e2_grad
            ),
            "stage_e1_grad_norm": float(
                stage_e1_grad
            ),
            "stage_e0_grad_norm": float(
                stage_e0_grad
            ),
            "dense_grad_norm": float(
                dense_grad
            ),
            "instance_proj_e2_grad_norm": float(
                e2_proj_grad
            ),
            "instance_proj_d1_grad_norm": float(
                d1_proj_grad
            ),
            "total_grad_norm_before_clip": float(
                total_grad_norm
            ),
            "finite_gradients": bool(
                finite
            ),
            "step_seconds": float(
                time.perf_counter()
                - step_start
            ),
            "peak_cuda_gib": float(
                torch.cuda.max_memory_allocated()
                / 1024**3
            ),
        })

        del (
            outputs,
            objective,
        )

        gc.collect()
        torch.cuda.empty_cache()

    elapsed = (
        time.perf_counter()
        - arm_start
    )

    metrics = pd.DataFrame(
        metric_rows
    )

    training = pd.DataFrame(
        train_rows
    )

    del (
        model,
        optimizer,
        scaler,
    )

    gc.collect()
    torch.cuda.empty_cache()

    return {
        "metrics": metrics,
        "training": training,
        "elapsed_s": float(
            elapsed
        ),
    }


ARM_RUNS = {}
metric_frames = []
training_frames = []
timing_rows = []

for arm, spec in ARM_SPECS.items():
    print(
        "\n"
        + "=" * 84
    )
    print(
        "RUNNING:",
        arm
    )
    print(
        "=" * 84
    )

    result = run_arm(
        arm,
        spec,
    )

    ARM_RUNS[arm] = result

    metric_frames.append(
        result["metrics"]
    )

    training_frames.append(
        result["training"]
    )

    timing_rows.append({
        "arm": arm,
        "seconds": result[
            "elapsed_s"
        ],
    })

    display(
        result["metrics"][
            [
                "arm",
                "step",
                "source9_internal_auc",
                "source9_internal_recall_0p5",
                "foreground_hard_dice",
                "e2_free_auc_mean",
                "d1_free_auc_mean",
                "d0_free_auc_mean",
                "e2_free_auc_min",
                "d1_free_auc_min",
            ]
        ]
    )

metrics_df = pd.concat(
    metric_frames,
    ignore_index=True,
)

training_df = pd.concat(
    training_frames,
    ignore_index=True,
)

timing_df = pd.DataFrame(
    timing_rows
)

metrics_df.to_csv(
    RUN_DIR
    / "metrics.csv",
    index=False,
)

training_df.to_csv(
    RUN_DIR
    / "training.csv",
    index=False,
)

timing_df.to_csv(
    RUN_DIR
    / "timing.csv",
    index=False,
)

print(
    "\nAll Notebook-23 arms complete."
)

# 16. Compare the final five-step states

All four arms began with identical step-30 base weights and identical notebook-local projection initialization.

In [ ]:
final_df = (
    metrics_df[
        metrics_df[
            "step"
        ]
        == TRAIN_STEPS
    ]
    .copy()
)

baseline0 = (
    metrics_df[
        (
            metrics_df[
                "arm"
            ]
            == "baseline"
        )
        & (
            metrics_df[
                "step"
            ]
            == 0
        )
    ]
    .iloc[0]
)

for column in (
    "source9_internal_auc",
    "foreground_hard_dice",
    "e2_free_auc_mean",
    "d1_free_auc_mean",
    "d0_free_auc_mean",
    "e2_free_auc_min",
    "d1_free_auc_min",
    "d0_free_auc_min",
):
    final_df[
        f"delta_{column}"
    ] = (
        final_df[column]
        - float(
            baseline0[column]
        )
    )

display(
    final_df[
        [
            "arm",
            "source9_internal_auc",
            "delta_source9_internal_auc",
            "source9_internal_recall_0p5",
            "foreground_hard_dice",
            "delta_foreground_hard_dice",
            "e2_free_auc_mean",
            "delta_e2_free_auc_mean",
            "d1_free_auc_mean",
            "delta_d1_free_auc_mean",
            "d0_free_auc_mean",
            "delta_d0_free_auc_mean",
            "e2_free_auc_min",
            "delta_e2_free_auc_min",
            "d1_free_auc_min",
            "delta_d1_free_auc_min",
            "d0_free_auc_min",
            "delta_d0_free_auc_min",
        ]
    ].sort_values(
        [
            "delta_d1_free_auc_mean",
            "delta_source9_internal_auc",
        ],
        ascending=False,
    )
)

final_df.to_csv(
    RUN_DIR
    / "final_comparison.csv",
    index=False,
)

# 17. Inspect whether the new objectives actually update the intended spatial modules

In [ ]:
first_step_gradients = (
    training_df[
        training_df[
            "from_step"
        ]
        == 0
    ][
        [
            "arm",
            "encoder_grad_norm",
            "stage_e2_grad_norm",
            "stage_e1_grad_norm",
            "stage_e0_grad_norm",
            "dense_grad_norm",
            "instance_proj_e2_grad_norm",
            "instance_proj_d1_grad_norm",
            "finite_gradients",
            "peak_cuda_gib",
            "step_seconds",
        ]
    ]
)

display(
    first_step_gradients
)

first_step_gradients.to_csv(
    RUN_DIR
    / "first_step_gradients.csv",
    index=False,
)

# 18. Automatic implementation decision

These thresholds are deliberately modest because this is only a five-step causal screen.

A permanent implementation is justified only when the proposed patch moves the **raw feature probe**, not merely its own auxiliary embedding geometry.

In [ ]:
def row_for(
    arm,
):
    rows = final_df[
        final_df[
            "arm"
        ]
        == arm
    ]

    if not len(rows):
        raise RuntimeError(
            f"Missing final row for {arm}"
        )

    return rows.iloc[0]


internal_row = row_for(
    "internal_boundary"
)

embedding_row = row_for(
    "instance_embedding"
)

combined_row = row_for(
    "combined"
)

boundary_validated = (
    float(
        internal_row[
            "delta_source9_internal_auc"
        ]
    )
    >= 0.01
)

embedding_mean_gain = 0.5 * (
    float(
        embedding_row[
            "delta_e2_free_auc_mean"
        ]
    )
    + float(
        embedding_row[
            "delta_d1_free_auc_mean"
        ]
    )
)

combined_mean_gain = 0.5 * (
    float(
        combined_row[
            "delta_e2_free_auc_mean"
        ]
    )
    + float(
        combined_row[
            "delta_d1_free_auc_mean"
        ]
    )
)

combined_worst_gain = min(
    float(
        combined_row[
            "delta_e2_free_auc_min"
        ]
    ),
    float(
        combined_row[
            "delta_d1_free_auc_min"
        ]
    ),
)

combined_boundary_gain = float(
    combined_row[
        "delta_source9_internal_auc"
    ]
)

combined_fg_delta = float(
    combined_row[
        "delta_foreground_hard_dice"
    ]
)

embedding_validated = (
    embedding_mean_gain >= 0.02
)

combined_identity_validated = (
    combined_mean_gain >= 0.02
)

combined_no_harm = (
    combined_fg_delta >= -0.02
)

combined_boundary_validated = (
    combined_boundary_gain >= 0.01
)

findings = []

if boundary_validated:
    findings.append(
        "Internal-boundary supervision reproduces the earlier causal improvement."
    )
else:
    findings.append(
        "Internal-boundary supervision did not reproduce a +0.01 AUC gain."
    )

if embedding_validated:
    findings.append(
        f"Instance-discriminative supervision improves raw E2/D1 linear mask separability "
        f"(mean gain={embedding_mean_gain:+.3f})."
    )
else:
    findings.append(
        f"Instance-discriminative supervision does not yet move raw E2/D1 separability "
        f"by +0.02 (mean gain={embedding_mean_gain:+.3f})."
    )

if combined_identity_validated:
    findings.append(
        f"The combined patch improves raw E2/D1 separability "
        f"(mean gain={combined_mean_gain:+.3f})."
    )

if combined_boundary_validated:
    findings.append(
        f"The combined patch also improves source-9 internal-boundary AUC "
        f"(gain={combined_boundary_gain:+.3f})."
    )

findings.append(
    f"Combined worst-cell E2/D1 minimum-AUC gain={combined_worst_gain:+.3f}."
)

if combined_no_harm:
    findings.append(
        f"Foreground hard Dice is preserved "
        f"(combined delta={combined_fg_delta:+.4f})."
    )
else:
    findings.append(
        f"Foreground hard Dice regresses materially "
        f"(combined delta={combined_fg_delta:+.4f})."
    )

if (
    combined_identity_validated
    and combined_boundary_validated
    and combined_no_harm
):
    verdict = "GREEN_IMPLEMENT_COMBINED_PATCH"
    next_action = (
        "Implement explicit internal-boundary target/loss plus E2/D1 "
        "instance-discriminative auxiliary supervision in the repository."
    )
elif (
    embedding_validated
    and combined_no_harm
):
    verdict = "YELLOW_IDENTITY_WORKS_TUNE_COMBINATION"
    next_action = (
        "The instance objective works, but the combined weighting needs one "
        "small weight-adjustment screen before permanent implementation."
    )
elif boundary_validated:
    verdict = "YELLOW_BOUNDARY_ONLY_CONFIRMED"
    next_action = (
        "Implement/retain the internal-boundary mechanism, but do not yet commit "
        "the instance embedding objective; inspect its gradients/margins or adjust "
        "only its loss formulation."
    )
else:
    verdict = "RED_DO_NOT_IMPLEMENT_PATCH"
    next_action = (
        "The notebook-only patch did not causally improve the intended raw spatial "
        "metrics. Do not implement it in source."
    )

print(
    "=" * 88
)
print(
    "NOTEBOOK 23 — SPATIAL SUPERVISION PATCH VERDICT"
)
print(
    "=" * 88
)
print(
    "Verdict:",
    verdict
)
print()

for index, finding in enumerate(
    findings,
    start=1,
):
    print(
        f"{index}. {finding}"
    )

print()
print(
    "Next action:",
    next_action
)

report = {
    "verdict": verdict,
    "next_action": next_action,
    "boundary_validated": bool(
        boundary_validated
    ),
    "embedding_validated": bool(
        embedding_validated
    ),
    "combined_identity_validated": bool(
        combined_identity_validated
    ),
    "combined_boundary_validated": bool(
        combined_boundary_validated
    ),
    "combined_no_harm": bool(
        combined_no_harm
    ),
    "embedding_mean_e2_d1_auc_gain": float(
        embedding_mean_gain
    ),
    "combined_mean_e2_d1_auc_gain": float(
        combined_mean_gain
    ),
    "combined_worst_e2_d1_auc_gain": float(
        combined_worst_gain
    ),
    "combined_boundary_auc_gain": float(
        combined_boundary_gain
    ),
    "combined_foreground_dice_delta": float(
        combined_fg_delta
    ),
    "findings": findings,
}

with (
    RUN_DIR
    / "verdict.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        report,
        handle,
        indent=2,
        default=float,
    )

# 19. Compact trajectory plots

In [ ]:
plot_metrics = metrics_df.copy()

fig, ax = plt.subplots(
    figsize=(9, 4)
)

for arm, group in plot_metrics.groupby(
    "arm"
):
    ax.plot(
        group["step"],
        group["source9_internal_auc"],
        marker="o",
        label=arm,
    )

ax.set_xlabel("micro-training step")
ax.set_ylabel(
    "source-9 internal-boundary AUC"
)
ax.set_title(
    "Boundary separation"
)
ax.legend()
plt.tight_layout()
plt.show()


probe_plot = plot_metrics[
    plot_metrics[
        "step"
    ].isin(
        sorted(
            LINEAR_PROBE_STEPS
        )
    )
]

fig, ax = plt.subplots(
    figsize=(9, 4)
)

for arm, group in probe_plot.groupby(
    "arm"
):
    ax.plot(
        group["step"],
        group["d1_free_auc_mean"],
        marker="o",
        label=arm,
    )

ax.set_xlabel("micro-training step")
ax.set_ylabel(
    "D1 free-vector held-out AUC"
)
ax.set_title(
    "Raw D1 cell-identity separability"
)
ax.legend()
plt.tight_layout()
plt.show()


fig, ax = plt.subplots(
    figsize=(9, 4)
)

for arm, group in probe_plot.groupby(
    "arm"
):
    ax.plot(
        group["step"],
        group["e2_free_auc_min"],
        marker="o",
        label=arm,
    )

ax.set_xlabel("micro-training step")
ax.set_ylabel(
    "E2 worst-cell held-out AUC"
)
ax.set_title(
    "Worst source-9 cell identity"
)
ax.legend()
plt.tight_layout()
plt.show()

# 20. Save manifest

This notebook never changes source files.

If the verdict is green, the next step is a small repository patch implementing the same target/head/loss contracts, followed first by the same 5–10 step acceptance screen — **not** an immediate two-hour staged overfit.

In [ ]:
manifest = {
    "checkpoint": str(
        STEP30_CHECKPOINT
    ),
    "train_steps": int(
        TRAIN_STEPS
    ),
    "arms": list(
        ARM_SPECS
    ),
    "loss_weights": {
        "internal_boundary": float(
            LAMBDA_INTERNAL_BOUNDARY
        ),
        "instance_e2": float(
            LAMBDA_INSTANCE_E2
        ),
        "instance_d1": float(
            LAMBDA_INSTANCE_D1
        ),
    },
    "instance_embedding": {
        "dim": int(
            INSTANCE_EMBED_DIM
        ),
        "samples_per_cell": int(
            INSTANCE_SAMPLES_PER_CELL
        ),
        "variance_margin": float(
            INSTANCE_VARIANCE_MARGIN
        ),
        "separation_margin": float(
            INSTANCE_SEPARATION_MARGIN
        ),
    },
    "files": [
        path.name
        for path in sorted(
            RUN_DIR.glob("*")
        )
    ],
}

with (
    RUN_DIR
    / "manifest.json"
).open(
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
    )

display(
    timing_df
)

print(
    json.dumps(
        manifest,
        indent=2,
    )
)

gc.collect()
torch.cuda.empty_cache()

print(
    "\nNotebook 23 complete. "
    "No repository source was modified and no full overfit was run."
)

# Interpretation guide

### `GREEN_IMPLEMENT_COMBINED_PATCH`

The two proposed changes survive a faithful notebook-only implementation:

- internal-boundary AUC improves;
- raw E2/D1 cell-identity AUC improves;
- foreground remains stable.

Proceed to implement the target, projection heads, criterion terms, curriculum weights, and tests in the repository.

### `YELLOW_IDENTITY_WORKS_TUNE_COMBINATION`

The instance-discriminative mechanism is useful, but the boundary + identity weights interfere slightly. Run only a tiny loss-weight adjustment; do not redesign the CNN.

### `YELLOW_BOUNDARY_ONLY_CONFIRMED`

The internal-boundary mechanism is real, but this particular discriminative embedding loss does not improve the raw representation quickly enough. Implementing the embedding head would be premature.

### `RED_DO_NOT_IMPLEMENT_PATCH`

Do not transfer this notebook patch into source. The next debugging target should be the spatial representation/decoder itself rather than adding these losses.